# YOLOv8-nano MRZ Detection Fine-tuning

COCO pretrained YOLOv8-nano を MRZ_Passport_dataset でファインチューニングし、
パスポート MRZ 領域の検出モデルを構築する。

- **データセット**: MRZ_Passport_dataset (Roboflow, CC BY 4.0)
- **画像数**: 57 (train 42, valid 10, test 5)
- **クラス**: 3 (MRZ_P, mrz1, mrz2)
- **ベースモデル**: YOLOv8-nano (COCO pretrained)

In [ ]:
# Cell 1: Setup
!pip install -q ultralytics
!nvidia-smi

In [ ]:
# Cell 2: データセット準備
import yaml
from pathlib import Path
import shutil

INPUT_DIR = Path("/kaggle/input/mrz-passport-dataset")
WORK_DIR = Path("/kaggle/working")

# data.yaml を読み込み、パスを Kaggle 環境用に書き換え
with open(INPUT_DIR / "data.yaml") as f:
    data_cfg = yaml.safe_load(f)

data_cfg["train"] = str(INPUT_DIR / "train" / "images")
data_cfg["val"] = str(INPUT_DIR / "valid" / "images")
data_cfg["test"] = str(INPUT_DIR / "test" / "images")

data_yaml_path = WORK_DIR / "data.yaml"
with open(data_yaml_path, "w") as f:
    yaml.dump(data_cfg, f, default_flow_style=False)

print(f"data.yaml saved to: {data_yaml_path}")
print(yaml.dump(data_cfg, default_flow_style=False))

# 画像数の確認
for split in ["train", "valid", "test"]:
    imgs = list((INPUT_DIR / split / "images").glob("*.[jp][pn][g]"))
    print(f"{split}: {len(imgs)} images")

In [ ]:
# Cell 3: YOLOv8-nano ファインチューニング v5
# v3→v5 変更点:
#   - single_cls=True: mrz1/mrz2 は各1サンプル(polygon形式)で学習不可 → 単一クラス化
#   - degrees=5→90: val の 5_jpg が90°回転パスポート(MRZ縦向き)でミス → 回転拡張で対応
# v4 で試した imgsz=1024, cos_lr, dropout, mosaic=0.5 等は悪化したため v3 ベースに戻す
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # COCO pretrained

results = model.train(
    data=str(data_yaml_path),
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    patience=20,
    save=True,
    project="runs",
    name="mrz-detect",
    # 転移学習設定
    freeze=10,
    single_cls=True,
    lr0=0.01,
    lrf=0.01,
    # 小規模データ用拡張
    augment=True,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=90.0,          # 90°回転パスポート対応
    translate=0.1,
    scale=0.5,
    flipud=0.0,
    fliplr=0.0,
)

In [ ]:
# Cell 4: 評価
best_model_path = Path(model.trainer.save_dir) / "weights" / "best.pt"
print(f"Best model: {best_model_path}")

best_model = YOLO(str(best_model_path))
metrics = best_model.val(data=str(data_yaml_path), single_cls=True)

print(f"mAP@0.5:      {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision:     {metrics.box.mp:.4f}")
print(f"Recall:        {metrics.box.mr:.4f}")

# クラスごとの AP (single_cls=True の場合は1クラスのみ)
for i in range(len(metrics.box.ap50)):
    print(f"  class {i}: AP50={metrics.box.ap50[i]:.4f}  AP={metrics.box.ap[i]:.4f}")

In [ ]:
# Cell 4.5: GT Coverage 評価 + マージンシミュレーション
# GT coverage = intersection / gt_area (1.0 = MRZ 全体が crop に含まれる)
# マージン付き bbox で coverage がどう改善するかも計測
import numpy as np
from PIL import Image

def load_yolo_labels(label_path, img_w, img_h):
    boxes = []
    for line in label_path.read_text().strip().split("\n"):
        if not line.strip():
            continue
        parts = line.strip().split()
        cx, cy, w, h = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
        x1 = (cx - w / 2) * img_w
        y1 = (cy - h / 2) * img_h
        x2 = (cx + w / 2) * img_w
        y2 = (cy + h / 2) * img_h
        boxes.append(np.array([x1, y1, x2, y2]))
    return boxes

def expand_box(box, margin_pct, img_w, img_h):
    """bbox を margin_pct 分だけ拡張（画像境界でクリップ）"""
    w = box[2] - box[0]
    h = box[3] - box[1]
    dx = w * margin_pct
    dy = h * margin_pct
    return np.array([
        max(0, box[0] - dx),
        max(0, box[1] - dy),
        min(img_w, box[2] + dx),
        min(img_h, box[3] + dy),
    ])

def compute_gt_coverage(gt_box, pred_box):
    x1 = max(gt_box[0], pred_box[0])
    y1 = max(gt_box[1], pred_box[1])
    x2 = min(gt_box[2], pred_box[2])
    y2 = min(gt_box[3], pred_box[3])
    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    gt_area = (gt_box[2] - gt_box[0]) * (gt_box[3] - gt_box[1])
    return intersection / gt_area if gt_area > 0 else 0.0

def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - intersection
    return intersection / union if union > 0 else 0.0

img_dir = INPUT_DIR / "valid" / "images"
label_dir = INPUT_DIR / "valid" / "labels"
image_paths = sorted(img_dir.glob("*.[jp][pn][g]"))
margins = [0.0, 0.05, 0.10, 0.15, 0.20]

# --- Per-image results (no margin) ---
print(f"{'Image':40s}  {'Coverage':>8s}  {'IoU':>6s}  {'Conf':>6s}  {'Status'}")
print("-" * 80)

# Collect raw predictions for margin simulation
all_results = []  # list of (img_path, img_w, img_h, gt_box, best_pred_xyxy)
for img_path in image_paths:
    img = Image.open(img_path)
    img_w, img_h = img.size
    label_path = label_dir / (img_path.stem + ".txt")
    gt_boxes = load_yolo_labels(label_path, img_w, img_h)
    if not gt_boxes:
        continue

    preds = best_model(img_path, verbose=False)[0]
    pred_boxes = preds.boxes

    for gt_box in gt_boxes:
        best_pred, best_iou, best_conf = None, 0.0, 0.0
        for pred_box in pred_boxes:
            xyxy = pred_box.xyxy[0].cpu().numpy()
            iou = compute_iou(gt_box, xyxy)
            if iou > best_iou:
                best_iou = iou
                best_pred = xyxy
                best_conf = float(pred_box.conf[0])

        cov = compute_gt_coverage(gt_box, best_pred) if best_pred is not None else 0.0
        all_results.append((img_path.stem, img_w, img_h, gt_box, best_pred, best_iou, best_conf))
        status = "OK" if cov >= 0.99 else ("PARTIAL" if cov > 0 else "MISS")
        print(f"{img_path.stem[:40]:40s}  {cov:8.4f}  {best_iou:6.4f}  {best_conf:6.2f}  [{status}]")

# --- Margin simulation ---
print(f"\n{'='*60}")
print(f"Margin Simulation: GT Coverage with expanded bbox")
print(f"{'='*60}")
print(f"{'Margin':>8s}  {'Mean':>8s}  {'Min':>8s}  {'>=0.99':>8s}  {'>=0.95':>8s}")
print("-" * 50)

for margin in margins:
    coverages = []
    for name, img_w, img_h, gt_box, pred, iou, conf in all_results:
        if pred is None:
            coverages.append(0.0)
            continue
        expanded = expand_box(pred, margin, img_w, img_h)
        cov = compute_gt_coverage(gt_box, expanded)
        coverages.append(cov)
    coverages = np.array(coverages)
    full = (coverages >= 0.99).sum()
    high = (coverages >= 0.95).sum()
    print(f"{margin:7.0%}  {coverages.mean():8.4f}  {coverages.min():8.4f}  {full:>5d}/10  {high:>5d}/10")

In [ ]:
# Cell 5: ONNX エクスポート
onnx_path = best_model.export(
    format="onnx",
    imgsz=320,
    simplify=True,
    opset=12,
    dynamic=False,
)
print(f"ONNX exported to: {onnx_path}")

In [ ]:
# Cell 6: ファイル保存 (Kaggle Output)
import shutil
from pathlib import Path

output_dir = Path("/kaggle/working")
save_dir = Path(model.trainer.save_dir)

# best.pt
best_pt = save_dir / "weights" / "best.pt"
shutil.copy2(best_pt, output_dir / "best.pt")
print(f"Copied {best_pt} -> {output_dir / 'best.pt'}")

# ONNX
onnx_src = Path(onnx_path)
onnx_dst = output_dir / "mrz_yolov8n.onnx"
shutil.copy2(onnx_src, onnx_dst)
print(f"Copied {onnx_src} -> {onnx_dst}")

# training results
results_csv = save_dir / "results.csv"
if results_csv.exists():
    shutil.copy2(results_csv, output_dir / "results.csv")
    print(f"Copied {results_csv} -> {output_dir / 'results.csv'}")

print("\nOutput files:")
for f in sorted(output_dir.glob("*")):
    if f.is_file():
        size_mb = f.stat().st_size / 1024 / 1024
        print(f"  {f.name} ({size_mb:.1f} MB)")